[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_10/notebook_10_3_gradcam_imaging.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 10.3: GradCAM for Medical Imaging

**Chapter 10: Interpretability and Explainability in Healthcare AI**

## Introduction: Seeing What CNNs See

A radiologist reviews an AI's chest X-ray interpretation:

```
AI Prediction: Pneumonia (92% confidence)
Recommendation: Initiate antibiotic therapy
```

**Critical question**: *Where* in the image does the AI see pneumonia?

**GradCAM (Gradient-weighted Class Activation Mapping)** answers this by generating visual heatmaps showing which regions of the image contribute most to the prediction.

### Why GradCAM for Medical Imaging?

1. **Visual validation**: Radiologists can verify AI is looking at clinically relevant regions
2. **Error detection**: If heatmap highlights wrong region, prediction is suspect
3. **Education**: Junior radiologists learn what features AI considers important
4. **Trust**: Transparency builds clinician confidence in AI assistance

### How GradCAM Works

1. **Forward pass**: Input image → CNN → prediction
2. **Backward pass**: Compute gradients of class score w.r.t. final conv layer
3. **Weight feature maps**: Each feature map weighted by its gradient importance
4. **Combine & upsample**: Weighted sum → ReLU → resize to input dimensions
5. **Overlay**: Heatmap on original image

---

## Part 1: Setup and Synthetic Medical Images

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
import tensorflow as tf
import cv2
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries imported")
print(f"TensorFlow version: {tf.__version__}")
print("\nThis notebook demonstrates GradCAM for explainable medical image AI.\n")

I0000 00:00:1795608962.333476    9998 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


✓ Libraries imported
TensorFlow version: 2.21.0

This notebook demonstrates GradCAM for explainable medical image AI.



I0000 00:00:1795608964.388065    9998 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Create Synthetic Medical Images

We'll create simplified "X-ray" images with pathological patterns.

In [2]:
def create_synthetic_xray_dataset(n_samples=1000, img_size=128):
    """
    Create synthetic X-ray-like images.

    Classes:
    - Normal: Uniform gray with texture; ~30% have a faint small benign density (e.g. a vessel seen end-on)
    - Pneumonia: Faint-to-moderate, large circular opacity in the lung field
    - Tumor: Smaller, denser nodule
    Lesion contrast and size vary and overlap between classes, so some cases are genuinely ambiguous
    (as in real radiographs): expect a realistic number of errors, not 100% accuracy.
    """
    images = []
    labels = []

    classes = ['Normal', 'Pneumonia', 'Tumor']

    for i in range(n_samples):
        # Base image (chest X-ray background)
        img = np.random.randint(100, 150, (img_size, img_size), dtype=np.uint8)

        # Add texture
        noise = np.random.randn(img_size, img_size) * 10
        img = img.astype(float) + noise
        img = np.clip(img, 0, 255).astype(np.uint8)

        # Determine class
        class_idx = np.random.choice([0, 1, 2])

        img = img.astype(float)
        y, x = np.ogrid[:img_size, :img_size]

        if class_idx == 0:  # Normal
            # Sometimes a faint, small benign density (a mimic of a small nodule)
            if np.random.random() < 0.3:
                cy, cx = np.random.randint(img_size//4, 3*img_size//4, 2)
                r = np.random.randint(4, 8)
                img[(x - cx)**2 + (y - cy)**2 <= r**2] += np.random.uniform(5, 20)

        elif class_idx == 1:  # Pneumonia
            # Add diffuse opacity (consolidation)
            center_y = np.random.randint(img_size//3, 2*img_size//3)
            center_x = np.random.randint(img_size//3, 2*img_size//3)
            radius = np.random.randint(10, 30)
            mask = (x - center_x)**2 + (y - center_y)**2 <= radius**2
            img[mask] += np.random.uniform(8, 30)  # faint-to-moderate opacity

        elif class_idx == 2:  # Tumor
            # Add dense nodule
            center_y = np.random.randint(img_size//4, 3*img_size//4)
            center_x = np.random.randint(img_size//4, 3*img_size//4)
            radius = np.random.randint(5, 13)
            mask = (x - center_x)**2 + (y - center_y)**2 <= radius**2
            img[mask] += np.random.uniform(15, 50)  # denser nodule

        # Normalize
        img = np.clip(img, 0, 255) / 255.0

        images.append(img)
        labels.append(class_idx)

    images = np.array(images)
    labels = np.array(labels)

    # Add channel dimension
    images = np.expand_dims(images, axis=-1)

    return images, labels, classes

# Create dataset
print("Generating synthetic medical images...\n")
X, y, class_names = create_synthetic_xray_dataset(n_samples=1000, img_size=128)

print(f"Dataset created:")
print(f"  Images: {X.shape}")
print(f"  Labels: {y.shape}")
print(f"  Classes: {class_names}")
print(f"\nClass distribution:")
for i, name in enumerate(class_names):
    print(f"  {name}: {(y == i).sum()} samples")

Generating synthetic medical images...



Dataset created:
  Images: (1000, 128, 128, 1)
  Labels: (1000,)
  Classes: ['Normal', 'Pneumonia', 'Tumor']

Class distribution:
  Normal: 328 samples
  Pneumonia: 326 samples
  Tumor: 346 samples


In [3]:
# Visualize examples
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, class_name in enumerate(class_names):
    # Find first example of this class
    idx = np.where(y == i)[0][0]

    axes[i].imshow(X[idx, :, :, 0], cmap='gray')
    axes[i].set_title(f'{class_name}', fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Synthetic Medical Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('synthetic_xrays.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📷 Synthetic X-rays generated for demonstration.")
print("   Note: Real medical imaging requires actual clinical data.\n")


📷 Synthetic X-rays generated for demonstration.
   Note: Real medical imaging requires actual clinical data.



In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train)} | Test: {len(X_test)}")

Training: 800 | Test: 200


## Part 2: Train CNN

In [5]:
# Build CNN
print("Building CNN model...\n")

model = models.Sequential([
    # Conv Block 1
    layers.Input(shape=(128, 128, 1)),
    # Standardise brightness (background ≈ 0.5, noise ≈ 0.07) so faint lesions are not swamped:
    # without this, training on low-contrast images often gets stuck predicting one class
    layers.Rescaling(1 / 0.07, offset=-0.5 / 0.07),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 4 (final conv layer for GradCAM)
    layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='final_conv'),
    layers.GlobalAveragePooling2D(),

    # Dense layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax', name='predictions')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Model architecture:")
model.summary()

Building CNN model...

✓ Model architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_conv (Conv2D)             │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,123 (1.61 MB)

 Trainable params: 421,123 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

Training is faster on a GPU. In Google Colab choose **Runtime → Change runtime type → GPU** (e.g. T4) and click **Save**.
The models here are small, so the notebook also runs on a laptop CPU in a few minutes.

In [6]:
# Train
print("\nTraining CNN...\n")

# Hold out a validation set from the TRAINING data. It is used to monitor training and to decide
# whether a run got stuck; the test set is used exactly once, after the final model is chosen.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=0, stratify=y_train)

for attempt in range(1, 4):
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=32,
        verbose=0
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    if val_acc > 0.75:   # a stuck run sits at ≈33% (one class) or ≈67% (two classes)
        break
    # Training sometimes gets stuck ignoring one or all classes (accuracy ≈ 33% or ≈ 67%).
    # Re-initialise the weights and try again rather than explaining a broken model.
    print(f"⚠️ Attempt {attempt}: training got stuck (validation accuracy {val_acc:.1%}). Retrying with fresh weights...")
    model = keras.models.clone_model(model)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Final evaluation on the untouched test set (once)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"✓ Model trained ({len(y_tr)} training, {len(y_val)} validation images)")
print(f"\nValidation Accuracy: {val_acc:.1%}")
print(f"Test Accuracy:       {test_acc:.1%}\n")
if val_acc <= 0.75:
    print("❌ Training got stuck 3 times: the Grad-CAM results below explain a broken model. Re-run this cell.\n")
elif test_acc > 0.98:
    print("⚠️ Near-perfect accuracy: this synthetic task is still much easier than real chest X-rays.\n")
else:
    print(f"ℹ️ {1 - test_acc:.1%} of test images are misclassified: faint opacities and benign densities")
    print("   overlap with real lesions here, so Part 6 has genuine errors to examine with Grad-CAM.\n")



Training CNN...



✓ Model trained (680 training, 120 validation images)

Validation Accuracy: 91.7%
Test Accuracy:       93.0%

ℹ️ 7.0% of test images are misclassified: faint opacities and benign densities
   overlap with real lesions here, so Part 6 has genuine errors to examine with Grad-CAM.



## Part 3: GradCAM Implementation

In [7]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate GradCAM heatmap.

    Parameters:
    - img_array: Input image (batch_size, height, width, channels)
    - model: Trained CNN
    - last_conv_layer_name: Name of last convolutional layer
    - pred_index: Class index to explain (None = use predicted class)

    Returns:
    - heatmap: GradCAM heatmap (0-1 range)
    """
    # Create model that outputs both predictions and final conv layer
    # (Keras 3: a Sequential model has .inputs/.outputs once built, but no single .output tensor)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.outputs[0]]
    )

    # Compute gradient of predicted class w.r.t. feature maps
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        if pred_index is None:
            pred_index = tf.argmax(predictions[0])

        class_channel = predictions[:, pred_index]

    # Gradient of class score w.r.t. feature maps
    grads = tape.gradient(class_channel, conv_outputs)

    # Global average pooling of gradients (importance weights)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply each feature map by its weight
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # ReLU (only positive contributions)
    heatmap = tf.maximum(heatmap, 0)

    # Normalize to [0, 1] (guard against an all-zero map)
    heatmap /= tf.math.reduce_max(heatmap) + 1e-8

    return heatmap.numpy()

print("✓ GradCAM function defined\n")

✓ GradCAM function defined



In [8]:
def display_gradcam(img, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """
    Overlay GradCAM heatmap on original image.
    """
    # Resize heatmap to match image size
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Convert heatmap to RGB
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Convert grayscale image to RGB
    if len(img.shape) == 2:
        img_rgb = np.stack([img]*3, axis=-1)
    else:
        img_rgb = img

    # Ensure same dtype
    img_rgb = np.uint8(255 * img_rgb)

    # Overlay
    superimposed = cv2.addWeighted(img_rgb, 1-alpha, heatmap, alpha, 0)

    return superimposed

def is_blank(heatmap, eps=1e-6):
    """True when the Grad-CAM map is (near) all-zero after ReLU: no positive evidence for the class.
    Its argmax would be pixel (0, 0), which is NOT a real peak, so blank maps must not be localised."""
    return float(np.max(heatmap)) < eps

def peak_on_lesion(img2d, heatmap, margin=8):
    """
    "Pointing game" check: does the heatmap's peak fall on the lesion?
    - The heatmap is upsampled to image size first (the last conv layer is only 16x16).
    - Lesion = clearly brighter area after smoothing (so single noisy pixels don't count),
      widened by `margin` pixels because the heatmap is coarse.
    Returns True/False, or None when the image has no bright lesion (Normal) or the heatmap is blank
    (check is_blank() first to tell the two apart).
    """
    if is_blank(heatmap):
        return None
    heat = cv2.resize(heatmap, (img2d.shape[1], img2d.shape[0]))
    peak = np.unravel_index(heat.argmax(), heat.shape)
    blur = cv2.GaussianBlur(img2d.astype(np.float32), (21, 21), 0)
    lesion = blur > np.median(blur) + 0.025   # brighter than this image's own background
    if not lesion.any():
        return None
    lesion = cv2.dilate(lesion.astype(np.uint8), np.ones((2 * margin + 1, 2 * margin + 1), np.uint8)) > 0
    return bool(lesion[peak])

print("✓ Visualization function defined\n")

✓ Visualization function defined



## Part 4: Generate GradCAM Explanations

In [9]:
# Select examples from each class
print("Generating GradCAM explanations...\n")

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
overlap_by_class = {}

for class_idx, class_name in enumerate(class_names):
    # Find example
    idx = np.where(y_test == class_idx)[0][0]
    img = X_test[idx]
    true_label = y_test[idx]

    # Predict
    img_array = np.expand_dims(img, axis=0)
    preds = model.predict(img_array, verbose=0)
    pred_class = np.argmax(preds[0])
    confidence = preds[0][pred_class]

    # Generate GradCAM
    heatmap = make_gradcam_heatmap(img_array, model, 'final_conv', pred_index=pred_class)
    if class_name == 'Normal':
        overlap_by_class[class_name] = None   # no pathology to localise (a benign density is not a target)
    else:
        overlap_by_class[class_name] = 'blank' if is_blank(heatmap) else peak_on_lesion(img[:, :, 0], heatmap)

    # Original image
    axes[class_idx, 0].imshow(img[:, :, 0], cmap='gray')
    axes[class_idx, 0].set_title(f'Original\n{class_name}', fontweight='bold')
    axes[class_idx, 0].axis('off')

    # Heatmap only
    axes[class_idx, 1].imshow(heatmap, cmap='jet')
    axes[class_idx, 1].set_title('GradCAM Heatmap', fontweight='bold')
    axes[class_idx, 1].axis('off')

    # Overlay
    overlay = display_gradcam(img[:, :, 0], heatmap, alpha=0.5)
    axes[class_idx, 2].imshow(overlay)
    axes[class_idx, 2].set_title(f'Overlay\nPred: {class_names[pred_class]} ({confidence:.1%})',
                                 fontweight='bold')
    axes[class_idx, 2].axis('off')

plt.suptitle('GradCAM Explanations for Medical Image Classification',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('gradcam_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 GradCAM Interpretation:")
print("\n  - Original: Input image")
print("  - Heatmap: Red = high importance, Blue = low importance")
print("  - Overlay: Heatmap superimposed on original")
print("\n  Does each heatmap's peak land on the lesion? (computed for the examples above)")
for name, hit in overlap_by_class.items():
    if hit == 'blank':
        print(f"  ⚠️ {name}: Grad-CAM map is blank (no positive evidence), so there is no peak to check")
    elif hit is None:
        print(f"  • {name}: no pathology to localise in a Normal image")
    elif hit:
        print(f"  ✓ {name}: the heatmap peak lies on the lesion")
    else:
        print(f"  ⚠️ {name}: the heatmap peak is OFF the lesion: the model may rely on other cues")
print("  Your turn: does that match what you see in the overlays?\n")

Generating GradCAM explanations...




📊 GradCAM Interpretation:

  - Original: Input image
  - Heatmap: Red = high importance, Blue = low importance
  - Overlay: Heatmap superimposed on original

  Does each heatmap's peak land on the lesion? (computed for the examples above)
  • Normal: no pathology to localise in a Normal image
  ✓ Pneumonia: the heatmap peak lies on the lesion
  ✓ Tumor: the heatmap peak lies on the lesion
  Your turn: does that match what you see in the overlays?



## Part 5: Clinical Validation

Check if GradCAM highlights clinically relevant regions.

In [10]:
def clinical_validation_report(model, X_test, y_test, class_names, n_samples=5):
    """
    Generate clinical validation report for GradCAM explanations.
    """
    print("="*80)
    print("CLINICAL VALIDATION: GradCAM Explanations")
    print("="*80 + "\n")

    for class_idx, class_name in enumerate(class_names):
        print(f"\n{class_name.upper()} Cases:\n")

        # Get samples
        indices = np.where(y_test == class_idx)[0][:n_samples]

        correct = 0
        total = 0

        for i, idx in enumerate(indices, 1):
            img = X_test[idx]
            img_array = np.expand_dims(img, axis=0)

            # Predict
            preds = model.predict(img_array, verbose=0)
            pred_class = np.argmax(preds[0])
            confidence = preds[0][pred_class]

            # GradCAM
            heatmap = make_gradcam_heatmap(img_array, model, 'final_conv', pred_index=pred_class)

            # Check if prediction is correct
            is_correct = (pred_class == class_idx)
            total += 1
            if is_correct:
                correct += 1

            # Analyze heatmap in IMAGE coordinates (the raw heatmap is only 16x16)
            heat = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
            peak_y, peak_x = np.unravel_index(heat.argmax(), heat.shape)
            near_edge = min(peak_y, peak_x, heat.shape[0] - 1 - peak_y, heat.shape[1] - 1 - peak_x) < 10
            hit = peak_on_lesion(img[:, :, 0], heatmap)

            # Clinical plausibility check (a blank map has no peak: its argmax (0,0) would look like an edge)
            if is_blank(heatmap):
                plausible = "⚠️ Blank Grad-CAM map (no positive evidence): cannot localise"
            elif near_edge:
                plausible = "⚠️ Peak at the image edge (artifact?)"
            elif class_name in ('Pneumonia', 'Tumor'):
                plausible = "✓ Peak on the lesion" if hit else "⚠️ Peak OFF the lesion"
            else:  # Normal: every heatmap is scaled to a peak of 1, so judge location, not height
                plausible = "✓ No edge artifact (a Normal image has no lesion to localise)"

            status = "✓" if is_correct else "✗"
            print(f"  {i}. Pred: {class_names[pred_class]} ({confidence:.1%}) {status}  |  {plausible}")

        accuracy = correct / total
        print(f"\n  Accuracy: {correct}/{total} ({accuracy:.1%})")

    print("\n" + "="*80)
    print("\n💡 Clinical Validation Criteria:")
    print("\n  ✓ Pneumonia: Heatmap should focus on consolidation region")
    print("  ✓ Tumor: Heatmap should be focal on nodule")
    print("  ✓ Normal: Heatmap should be diffuse (no focal pathology)")
    print("  ✗ Red flag: Heatmap highlights image edges or artifacts\n")

# Run validation
clinical_validation_report(model, X_test, y_test, class_names, n_samples=5)

CLINICAL VALIDATION: GradCAM Explanations


NORMAL Cases:

  1. Pred: Normal (99.2%) ✓  |  ✓ No edge artifact (a Normal image has no lesion to localise)
  2. Pred: Tumor (51.4%) ✗  |  ✓ No edge artifact (a Normal image has no lesion to localise)


  3. Pred: Normal (99.3%) ✓  |  ✓ No edge artifact (a Normal image has no lesion to localise)
  4. Pred: Normal (99.3%) ✓  |  ✓ No edge artifact (a Normal image has no lesion to localise)
  5. Pred: Normal (99.2%) ✓  |  ✓ No edge artifact (a Normal image has no lesion to localise)

  Accuracy: 4/5 (80.0%)

PNEUMONIA Cases:



  1. Pred: Tumor (90.5%) ✗  |  ✓ Peak on the lesion
  2. Pred: Pneumonia (100.0%) ✓  |  ⚠️ Blank Grad-CAM map (no positive evidence): cannot localise
  3. Pred: Pneumonia (100.0%) ✓  |  ✓ Peak on the lesion


  4. Pred: Pneumonia (100.0%) ✓  |  ✓ Peak on the lesion
  5. Pred: Pneumonia (91.5%) ✓  |  ✓ Peak on the lesion

  Accuracy: 4/5 (80.0%)

TUMOR Cases:

  1. Pred: Tumor (98.7%) ✓  |  ✓ Peak on the lesion


  2. Pred: Tumor (86.6%) ✓  |  ✓ Peak on the lesion
  3. Pred: Tumor (100.0%) ✓  |  ✓ Peak on the lesion
  4. Pred: Tumor (99.8%) ✓  |  ✓ Peak on the lesion


  5. Pred: Tumor (99.7%) ✓  |  ✓ Peak on the lesion

  Accuracy: 5/5 (100.0%)


💡 Clinical Validation Criteria:

  ✓ Pneumonia: Heatmap should focus on consolidation region
  ✓ Tumor: Heatmap should be focal on nodule
  ✓ Normal: Heatmap should be diffuse (no focal pathology)
  ✗ Red flag: Heatmap highlights image edges or artifacts



## Part 6: Detecting Model Errors with GradCAM

In [11]:
# Find misclassified examples
print("Analyzing misclassifications with GradCAM...\n")

# Get predictions
preds_all = model.predict(X_test, verbose=0)
pred_classes = np.argmax(preds_all, axis=1)

# Find errors
errors = np.where(pred_classes != y_test)[0]

print(f"Found {len(errors)} misclassifications out of {len(y_test)} test samples ({len(errors)/len(y_test):.1%})\n")

if len(errors) > 0:
    # Analyze first few errors
    fig, axes = plt.subplots(min(3, len(errors)), 3, figsize=(12, 4*min(3, len(errors))))

    if len(errors) == 1:
        axes = axes.reshape(1, -1)

    for i, err_idx in enumerate(errors[:3]):
        img = X_test[err_idx]
        true_label = y_test[err_idx]
        pred_label = pred_classes[err_idx]

        # Generate GradCAM
        img_array = np.expand_dims(img, axis=0)
        heatmap = make_gradcam_heatmap(img_array, model, 'final_conv', pred_index=pred_label)

        # Visualize
        axes[i, 0].imshow(img[:, :, 0], cmap='gray')
        axes[i, 0].set_title(f'Original\nTrue: {class_names[true_label]}', fontweight='bold')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(heatmap, cmap='jet')
        axes[i, 1].set_title('GradCAM', fontweight='bold')
        axes[i, 1].axis('off')

        overlay = display_gradcam(img[:, :, 0], heatmap, alpha=0.5)
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Predicted: {class_names[pred_label]}\n(Incorrect)',
                            fontweight='bold', color='red')
        axes[i, 2].axis('off')

    plt.suptitle('Misclassifications: GradCAM Reveals Model Errors',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_errors.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n💡 Error Analysis:")
    print("\n  GradCAM helps identify WHY the model made mistakes:")
    print("  • Is the model looking at the right region?")
    print("  • Is the pathology subtle or ambiguous?")
    print("  • Is the model focusing on artifacts?")
    print("\n  This guides model improvement and retraining.\n")
else:
    print("\n✓ No misclassifications found (perfect accuracy on test set)\n")

Analyzing misclassifications with GradCAM...



Found 14 misclassifications out of 200 test samples (7.0%)




💡 Error Analysis:

  GradCAM helps identify WHY the model made mistakes:
  • Is the model looking at the right region?
  • Is the pathology subtle or ambiguous?
  • Is the model focusing on artifacts?

  This guides model improvement and retraining.



## Part 7: Spurious Correlation Detection (the "Hospital Marker" Shortcut)

The chapter warns that imaging models can learn **hospital identifiers or equipment markers in image corners**
instead of pathology. Here we create that failure on purpose:

1. **Synthetic** images: Normal vs. Pneumonia, where the pneumonia opacity is deliberately **faint**.
2. In the **training** data, most Pneumonia images come from "Hospital A", whose scanner stamps a small bright
   marker in the **top-left corner**. Few Normal images carry it.
3. We train a CNN, then ask Grad-CAM *where* it looks, and test it at a "new hospital" where the marker is
   absent, moved, or appears on the wrong patients.

If accuracy is high in-distribution but collapses when the marker changes, the model learned the shortcut.

In [12]:
IMG = 64
MARK = (slice(3, 11), slice(3, 11))   # 8x8 "hospital marker" in the top-left corner

def make_marker_dataset(n, p_marker_pneu, p_marker_norm, seed, marker_pos='top_left'):
    """Synthetic Normal (0) vs Pneumonia (1) images with an optional corner marker."""
    rs = np.random.RandomState(seed)
    X = np.clip(rs.normal(0.5, 0.05, (n, IMG, IMG)), 0, 1)
    y = rs.randint(0, 2, n)
    yy, xx = np.ogrid[:IMG, :IMG]
    for i in range(n):
        if y[i] == 1:   # faint opacity in the lung field (the real signal)
            cy, cx, r = rs.randint(20, 44), rs.randint(20, 44), rs.randint(8, 14)
            X[i][(xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2] += 0.06
        if rs.rand() < (p_marker_pneu if y[i] == 1 else p_marker_norm):
            if marker_pos == 'top_left':
                X[i][MARK] = 1.0
            else:       # same marker, opposite corner
                X[i][-11:-3, -11:-3] = 1.0
    return np.clip(X, 0, 1)[..., None].astype('float32'), y

# Training data: marker on 90% of Pneumonia and 10% of Normal images (site/selection bias)
Xs_tr, ys_tr = make_marker_dataset(1500, 0.90, 0.10, seed=1)
Xs_te, ys_te = make_marker_dataset(400, 0.90, 0.10, seed=2)                        # same hospital mix
Xs_none, ys_none = make_marker_dataset(400, 0.0, 0.0, seed=2)                      # new site: no marker
Xs_swap, ys_swap = make_marker_dataset(400, 0.10, 0.90, seed=2)                    # marker on the wrong patients
Xs_move, ys_move = make_marker_dataset(400, 0.90, 0.10, seed=2, marker_pos='bottom_right')  # marker moved

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, img) in zip(axes, [('Normal (no marker)', Xs_tr[(ys_tr == 0)][0]),
                                   ('Pneumonia + marker', Xs_tr[(ys_tr == 1) & (Xs_tr[:, 5, 5, 0] == 1)][0]),
                                   ('Pneumonia, no marker', Xs_tr[(ys_tr == 1) & (Xs_tr[:, 5, 5, 0] < 1)][0])]):
    ax.imshow(img[:, :, 0], cmap='gray', vmin=0.3, vmax=1.0)
    ax.set_title(title, fontweight='bold'); ax.axis('off')
plt.suptitle('Synthetic training images with a corner "hospital marker"', fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Training: {len(ys_tr)} images | marker on {(Xs_tr[ys_tr == 1, 5, 5, 0] == 1).mean():.0%} of Pneumonia, "
      f"{(Xs_tr[ys_tr == 0, 5, 5, 0] == 1).mean():.0%} of Normal")


Training: 1500 images | marker on 90% of Pneumonia, 11% of Normal


In [13]:
tf.random.set_seed(0)
shortcut_model = models.Sequential([
    layers.Input(shape=(IMG, IMG, 1)),
    layers.Conv2D(16, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same', name='sp_conv'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(2, activation='softmax'),
])
shortcut_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
shortcut_model.fit(Xs_tr, ys_tr, epochs=12, batch_size=32, verbose=0)

def acc(Xa, ya):
    return float((shortcut_model.predict(Xa, verbose=0).argmax(1) == ya).mean())

results = {
    'Same hospital mix (marker as in training)': acc(Xs_te, ys_te),
    'New site: no markers at all': acc(Xs_none, ys_none),
    'Marker moved to bottom-right corner': acc(Xs_move, ys_move),
    'Marker on the wrong patients (90% Normal)': acc(Xs_swap, ys_swap),
}
print("Accuracy of the same model on four test sets (identical images, only the marker differs):\n")
for k, v in results.items():
    print(f"  {k:<45} {v:.1%}")


Accuracy of the same model on four test sets (identical images, only the marker differs):

  Same hospital mix (marker as in training)     90.5%
  New site: no markers at all                   47.0%
  Marker moved to bottom-right corner           47.5%
  Marker on the wrong patients (90% Normal)     10.5%


In [14]:
# Where does Grad-CAM look? Share of heatmap peaks that fall in the marker corner (computed)
CORNER = 16   # top-left 16x16 region = 6.25% of the image area (chance level for a random peak)
def peak_in_corner(img):
    """(True/False, heatmap), or (None, heatmap) for a blank map, whose argmax (0,0) is not a real peak."""
    heat = make_gradcam_heatmap(img[None], shortcut_model, 'sp_conv', pred_index=1)  # explain 'Pneumonia'
    if is_blank(heat):
        return None, cv2.resize(heat, (IMG, IMG))
    heat = cv2.resize(heat, (IMG, IMG))
    py, px = np.unravel_index(heat.argmax(), heat.shape)
    return (py < CORNER and px < CORNER), heat

marked_pneu = np.where((ys_te == 1) & (Xs_te[:, 5, 5, 0] == 1))[0][:40]
peaks = [peak_in_corner(Xs_te[i]) for i in marked_pneu]
valid = [p[0] for p in peaks if p[0] is not None]
n_blank = len(peaks) - len(valid)
corner_share = np.mean(valid) if valid else float('nan')
print(f"Blank Grad-CAM maps (excluded from localisation): {n_blank} of {len(peaks)}")
print(f"Grad-CAM peak inside the marker corner for {corner_share:.0%} of {len(valid)} non-blank maps "
      f"({len(marked_pneu)} marked Pneumonia images)")
print(f"(a peak placed at random would land there {CORNER**2 / IMG**2:.0%} of the time)\n")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k in range(4):
    img = Xs_te[marked_pneu[k]]
    axes[0, k].imshow(img[:, :, 0], cmap='gray', vmin=0.3, vmax=1.0); axes[0, k].axis('off')
    axes[0, k].set_title('Pneumonia + marker', fontweight='bold')
    axes[1, k].imshow(display_gradcam(img[:, :, 0], peaks[k][1], alpha=0.5)); axes[1, k].axis('off')
    axes[1, k].set_title('Grad-CAM (Pneumonia)', fontweight='bold')
plt.suptitle('Shortcut model: what does Grad-CAM highlight?', fontweight='bold')
plt.tight_layout(); plt.show()

in_dist = results['Same hospital mix (marker as in training)']
worst_name = min(results, key=results.get)
drop = in_dist - results[worst_name]
print("🔎 Verdict (computed):")
if not valid:
    print("  All Grad-CAM maps were blank: Grad-CAM cannot say where this model looks.")
elif corner_share >= 0.5:
    print(f"  ⚠️ Grad-CAM lights up the corner marker in {corner_share:.0%} of cases: the model looks at the marker, not the lung.")
else:
    print(f"  Grad-CAM points at the marker corner in only {corner_share:.0%} of cases.")
if drop >= 0.15:
    print(f"  ⚠️ Accuracy falls from {in_dist:.1%} to {results[worst_name]:.1%} ('{worst_name}').")
    print("     The model learned 'which hospital' rather than 'which disease'. It would fail at a new site.")
else:
    print(f"  Accuracy holds up across marker changes (largest drop {drop:.1%}): the shortcut was not decisive here.")
moved = results['Marker moved to bottom-right corner']
if abs(moved - in_dist) < 0.05:
    print(f"  Moving the marker to another corner barely matters ({moved:.1%}): convolution + global average")
    print("     pooling detect the marker wherever it is. The shortcut is 'bright square present', not its position.")
else:
    print(f"  Moving the marker to another corner changes accuracy to {moved:.1%}: the model also learned its position.")
print("\n💡 Lesson: in-distribution accuracy cannot reveal this. Grad-CAM plus a test where the marker")
print("   changes can. Before deployment, check heatmaps AND test on data from other sites/scanners.\n")


Blank Grad-CAM maps (excluded from localisation): 0 of 40
Grad-CAM peak inside the marker corner for 100% of 40 non-blank maps (40 marked Pneumonia images)
(a peak placed at random would land there 6% of the time)

🔎 Verdict (computed):
  ⚠️ Grad-CAM lights up the corner marker in 100% of cases: the model looks at the marker, not the lung.
  ⚠️ Accuracy falls from 90.5% to 10.5% ('Marker on the wrong patients (90% Normal)').
     The model learned 'which hospital' rather than 'which disease'. It would fail at a new site.
  Moving the marker to another corner changes accuracy to 47.5%: the model also learned its position.

💡 Lesson: in-distribution accuracy cannot reveal this. Grad-CAM plus a test where the marker
   changes can. Before deployment, check heatmaps AND test on data from other sites/scanners.



## Summary and Key Takeaways

### What We Learned

1. **GradCAM Mechanism**:
   - Uses gradients to weight feature maps by importance
   - Generates visual heatmap showing influential regions
   - Works with any CNN architecture

2. **Clinical Applications**:
   - **Validation**: Verify AI looks at clinically relevant regions
   - **Error detection**: Identify when model focuses on wrong features
   - **Education**: Teaching tool for radiology residents
   - **Trust**: Transparency builds clinician confidence

3. **Implementation**:
   - Requires access to model internals (last conv layer)
   - Fast: milliseconds per image
   - Easy to integrate into clinical workflows

4. **Validation Criteria**:
   ```
   ✓ Pneumonia: Heatmap highlights consolidation
   ✓ Tumor: Focal activation on nodule
   ✓ Normal: Diffuse attention, no focal hotspots
   ✗ Red flags: Edge artifacts, wrong anatomy
   ```

### Best Practices

✅ **Always validate with radiologists**: GradCAM doesn't guarantee clinical correctness  
✅ **Check multiple examples**: Don't rely on single heatmap  
✅ **Compare across classes**: Ensure different pathologies have different patterns  
✅ **Watch for artifacts**: Model may learn shortcuts (e.g., image borders)  
✅ **Use correct conv layer**: Last conv layer usually best for localization  

### Limitations

⚠️ **Coarse localization**: GradCAM heatmaps are lower resolution than input  
⚠️ **Multiple lesions**: May miss secondary findings  
⚠️ **Not pixel-precise**: Better for region-level interpretation  
⚠️ **Requires careful validation**: Heatmap doesn't prove clinical correctness  

### Clinical Deployment

**Integration Workflow**:
```
1. Radiologist reviews image
2. AI generates prediction + GradCAM overlay
3. Radiologist checks:
   - Is highlighted region plausible?
   - Does it match their clinical assessment?
4. If yes: AI confirms diagnosis
   If no: Re-examine image, trust human judgment
5. Report includes AI prediction + explanation
```

**Real-World Impact**:
- Trust: a convincing heatmap can raise clinicians' trust, which only helps when the map is right (see the pointing-game results above)
- Error detection: GradCAM caught models learning spurious features (e.g., hospital watermarks)
- Education: heatmaps help discuss what a model looks at, but they are not evidence that it reasons like a radiologist

---

## Exercises

1. **How strong must the marker bias be?**: In Part 7, change the training call `make_marker_dataset(1500, 0.90, 0.10, seed=1)` to marker rates of 0.70/0.30 and 0.55/0.45, retrain `shortcut_model`, and rerun the four test sets and the corner-peak check. At what imbalance does the model stop relying on the marker?

2. **Remove the shortcut (deeper)**: Retrain the shortcut model on data where the marker appears on a random half of all images regardless of class (or in a random corner). Check that (a) accuracy on "New site: no markers" recovers and (b) Grad-CAM peaks leave the marker corner. Then make the opacity fainter (the `+= 0.06` in `make_marker_dataset`) and see how faint the real signal can get before the model fails without the marker.

3. **Which layer to explain?**: Call `make_gradcam_heatmap` with an earlier convolutional layer (names from `model.summary()`) instead of `'final_conv'` for the Part 4 examples. Are the maps sharper or noisier, and does `peak_on_lesion` still find the lesion?

4. **Grad-CAM on the errors**: For each of the 16 misclassified test images in Part 6, record `peak_on_lesion` and `is_blank`. Are most errors "looked at the lesion but chose the wrong class" (for example Pneumonia predicted as Tumor) or "looked elsewhere"? Also count how many correct, highly confident predictions give a blank map, as Pneumonia case 2 did in Part 5. Hint: a saturated softmax gives near-zero gradients.

5. **Reflection: Jamal's seven findings**: In Chapter 3, the nodule AI flagged seven findings on Jamal's screening CT; one was a real 8 mm nodule and the others were vessels seen end-on, scar and calcifications. A heatmap for finding 2 would sit neatly on the vessel. What does a well-placed heatmap tell the radiologist, and what does it not? Would heatmaps help dismiss false positives faster, or encourage confirmation bias?

---

**Next**: Notebook 10.4 covers counterfactual explanations ("what if" scenarios for clinical decisions).

*"A radiologist can't trust what they can't see. GradCAM makes AI's vision visible."*